# KinderCompass knowledge-graph build

This notebook demonstrates the supported Neo4j loader in `SystemCode/src/scripts/build_knowledge_graph.py`. Normal runs update the graph without clearing it. Database deletion requires an explicit opt-in.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

start_dir = Path.cwd().resolve()
repo_root = next((path for path in (start_dir, *start_dir.parents) if (path / 'SystemCode').is_dir()), None)
if repo_root is None:
    raise RuntimeError('Could not locate the KinderCompass repository root')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / '.env')
from SystemCode.src.scripts.build_knowledge_graph import DEFAULT_INPUT, build_graph, get_driver, load_catalogue

In [ ]:
configuration = {
    'neo4j_uri_configured': bool(os.getenv('NEO4J_URI')),
    'neo4j_user_configured': bool(os.getenv('NEO4J_USERNAME')),
    'neo4j_password_configured': bool(os.getenv('NEO4J_PASSWORD')),
    'input': str(DEFAULT_INPUT),
}
configuration

## Load and validate the processed catalogue

This step is local and does not connect to or modify Neo4j.

In [ ]:
records = load_catalogue(DEFAULT_INPUT)
print(f'Validated {len(records):,} schools')
records[:2]

## Update Neo4j

The default below is non-destructive: it upserts preschool nodes and synchronizes their location and care-level relationships. Set `clear_existing = True` only when you intentionally want to delete every node before rebuilding the graph.

In [ ]:
clear_existing = False

with get_driver() as driver:
    result = build_graph(records, driver, clear_existing=clear_existing)
result